In [1]:
# Cell 1 — Imports
from pathlib import Path
import sys

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_config, load_model
from gcamp_analysis.experiments.tree import ExperimentTreeBuilder, is_video_dir, print_tree
from gcamp_analysis.video_runner import VideoPipelineRunner
from gcamp_analysis.experiments.processor import ExperimentProcessor
from gcamp_analysis.experiments.io import save_comparisons, save_treatment_comparisons

In [2]:
config_path = PROJECT_ROOT / "config" / "notebook_config.yaml"
config = load_config(config_path)

print(f"Config: {config_path}")

Config: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\notebook_config.yaml


In [3]:
roi_model, roi_cfg = load_model(config["models"], which="roi")
spike_model, spike_cfg = load_model(config["models"], which="spike")

models = {
    "roi": roi_model,
    "roi_config": roi_cfg,
    "spike": spike_model,
    "spike_config": spike_cfg,
}

runner = VideoPipelineRunner.build(config, models)

print(f"ROI model:   {type(roi_model).__name__}")
print(f"Spike model: {type(spike_model).__name__}")

ROI model:   RandomForestClassifier
Spike model: LogisticRegression


In [4]:
EXPERIMENT_ROOT = Path(r"C:\Users\mzinn1\Desktop\DailyRecordings_LateTimepoints")  # TODO: change per experiment
assert EXPERIMENT_ROOT.exists(), f"Experiment root not found: {EXPERIMENT_ROOT}"

builder = ExperimentTreeBuilder(is_video_dir=is_video_dir)
tree = builder.build(EXPERIMENT_ROOT)
print_tree(tree)

└── DailyRecordings_LateTimepoints
    ├── BP
    │   ├── 1-1_Day13
    │   ├── 1-1_Day21
    │   ├── 1-1_Day28
    │   ├── 1-2_Day13
    │   ├── 1-2_Day21
    │   ├── 1-2_Day28
    │   ├── 1-3_Day13
    │   ├── 1-3_Day21
    │   ├── 1-3_Day28
    │   ├── 1-4_Day13
    │   ├── 1-4_Day21
    │   ├── 1-4_Day28
    │   ├── 1-5_Day13
    │   ├── 1-5_Day21
    │   └── 1-5_Day28
    ├── IOBP
    │   ├── 13-1_IOBP
    │   ├── 13-2_IOBP
    │   ├── 13-3_IOBP
    │   ├── 13-4_IOBP
    │   ├── 13-5_IOBP
    │   ├── 21-1_IOBP
    │   ├── 21-2_IOBP
    │   ├── 21-3_IOBP
    │   ├── 21-4_IOBP
    │   ├── 21-5_IOBP
    │   ├── 28-1_IOBP
    │   ├── 28-2_IOBP
    │   ├── 28-3_IOBP
    │   ├── 28-4_IOBP
    │   └── 28-5_IOBP
    └── _nonrecording
        ├── Day 13
        │   └── trash
        ├── Day 21
        └── Day 28


In [5]:
processor = ExperimentProcessor(
    runner=runner,
    output_root=EXPERIMENT_ROOT,
)
processor.process_tree(tree, verbose=True)


 Processing: 1-1_Day13
  Traces: 1610 ROIs, 1818 frames @ 15.0 Hz
  ROI filter: 889/1610 kept (55.2%)
  Spikes: 14911/36472 kept | neurons 889 -> 889
  Grouping (combined): | combined=23

 Processing: 1-1_Day21
  Traces: 1163 ROIs, 1818 frames @ 15.0 Hz
  ROI filter: 2/1163 kept (0.2%)
  Spikes: 14/99 kept | neurons 2 -> 2
  Grouping (combined): | combined=0

 Processing: 1-1_Day28
  Traces: 492 ROIs, 1818 frames @ 15.0 Hz
  ROI filter: 1/492 kept (0.2%)
  Spikes: 15/51 kept | neurons 1 -> 1
  <2 neurons with spikes - skipping grouping.

 Processing: 1-2_Day13
  Traces: 867 ROIs, 1818 frames @ 15.0 Hz
  ROI filter: 457/867 kept (52.7%)
  Spikes: 6757/18483 kept | neurons 457 -> 457
  Grouping (combined): | combined=15

 Processing: 1-2_Day21
  Traces: 482 ROIs, 1818 frames @ 15.0 Hz
  ROI filter: 0/482 kept (0.0%)
  No ROIs kept - skipping spikes and grouping.

 Processing: 1-2_Day28
  Traces: 307 ROIs, 1818 frames @ 15.0 Hz
  ROI filter: 2/307 kept (0.7%)
  Spikes: 22/78 kept | neuro

In [6]:
 
sibling_tables = processor.compare_siblings(tree)

for node_path, df in sibling_tables.items():
    if len(df) >= 2:
        print(f"\nNode: {node_path}")
        print(df.to_string(index=False))


Node: C:\Users\mzinn1\Desktop\DailyRecordings_LateTimepoints
child  n_videos  n_neurons  n_groups_combined  mean_group_size_combined  median_group_size_combined  mean_group_corr_combined  mean_spikes_per_group_combined  frac_grouped  frac_ungrouped  decay_tau_seconds_mean_unweighted  half_max_width_seconds_mean_unweighted  rise_slope_hz_mean_unweighted  decay_tau_seconds_mean_weighted  half_max_width_seconds_mean_weighted  rise_slope_hz_mean_weighted  decay_tau_seconds_mean_grouped  half_max_width_seconds_mean_grouped  rise_slope_hz_mean_grouped  decay_tau_seconds_mean_ungrouped  half_max_width_seconds_mean_ungrouped  rise_slope_hz_mean_ungrouped  spike_frequency_mean_unweighted  spike_frequency_mean_weighted  spike_frequency_mean_grouped  spike_frequency_mean_ungrouped  decay_tau_seconds_var_unweighted  decay_tau_seconds_within_unweighted  decay_tau_seconds_between_unweighted  half_max_width_seconds_var_unweighted  half_max_width_seconds_within_unweighted  half_max_width_seconds_betw

In [7]:
save_comparisons(
    root=tree,
    sibling_tables=sibling_tables,
    output_subdir="metrics",
    filename="sibling_comparisons.xlsx",
)

save_treatment_comparisons(tree)